In [83]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    # Python
    random.seed(seed)
    # Numpy
    np.random.seed(seed)
    # PyTorch (CPU)
    torch.manual_seed(seed)
    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo (importante!)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Algumas libs usam isso
    os.environ["PYTHONHASHSEED"] = str(seed)
    # Para transformers (às vezes ajuda)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"Seed definida como {seed}")

seed_everything(42)

Seed definida como 42


In [84]:
import numpy as np
import os
from datetime import datetime
import pandas as pd
from tqdm import tqdm

import datasets
import evaluate

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel

from sklearn.metrics import f1_score, accuracy_score

In [85]:
# Mapeamento real do dataset de treino (ordenado por frequência, não alfabeticamente)
DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark',
                 'advcl', 'case', 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod',
                 'flat:name', 'ccomp', 'cop', 'acl', 'nummod', 'acl:relcl',
                 'ccomp:speech', 'parataxis', 'csubj', 'aux:pass', 'appos', 'fixed',
                 'nsubj:pass', 'aux', 'nsubj:outer', 'obl:agent', 'expl:impers',
                 'expl', 'discourse', 'orphan', 'dislocated', 'flat', 'flat:foreign',
                 'iobj', 'vocative', 'csubj:outer', 'list', 'reparandum', 'csubj:pass']

DEPREL_LABELS_TO_IDX = {i: idx for idx, i in enumerate(DEPREL_LABELS)}
IDX_TO_DEPREL_LABELS = {i: j for j, i in DEPREL_LABELS_TO_IDX.items()}

# ── Para trocar de modelo, altere apenas estas variáveis ──────────────────────
# mBERT           : MODEL_NAME = 'google-bert/bert-base-multilingual-cased'  FOLD = ?
# BERTimbau-Base  : MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'      FOLD = ?
# BERTimbau-Large : MODEL_NAME = 'neuralmind/bert-large-portuguese-cased'     FOLD = ?
# ModernJabutica  : MODEL_NAME = 'amadeusai/modernJabuticaBERT-Base-1k'      FOLD = ?
# (consulte ablacao_linear_results.jsonl para identificar o melhor fold por LAS)
MODEL_NAME = 'amadeusai/modernJabuticaBERT-Base-1k'
FOLD       = 0  # Melhor fold por LAS de validação

FINETUNED_MODEL_PATH = f'./best_models_ablacao_linear/{MODEL_NAME.replace("/", "_")}'
TOKENIZER_NAME       = MODEL_NAME
# ──────────────────────────────────────────────────────────────────────────────

In [86]:
from transformers import PreTrainedModel, AutoModel
from typing import Optional

# Ablação: apenas DEPREL + HEAD (sem UPOS) — cabeças lineares
class MultiTaskSentencePredictionEncoderAblacao(PreTrainedModel):
    _tied_weights_keys = []
    all_tied_weights_keys = {}

    def __init__(self, config, num_deprel_labels, num_head_labels=200):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.num_head_labels   = num_head_labels

        self.bert = AutoModel.from_config(config)

        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
        self.head_classifier   = nn.Linear(config.hidden_size, num_head_labels)

        classifier_dropout = (
            getattr(config, 'classifier_dropout', None)
            or getattr(config, 'hidden_dropout_prob', 0.1)
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        deprel_label:   Optional[torch.Tensor] = None,
        head_label:     Optional[torch.Tensor] = None,
    ):
        try:
            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        sequence_output = self.dropout(outputs[0])

        logits_deprel = self.deprel_classifier(sequence_output)
        logits_head   = self.head_classifier(sequence_output)

        loss = None
        if deprel_label is not None and head_label is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = (
                loss_fct(logits_deprel.view(-1, self.num_deprel_labels), deprel_label.view(-1))
                + loss_fct(logits_head.view(-1, self.num_head_labels),   head_label.view(-1))
            )

        if loss is not None:
            return (loss, logits_deprel, logits_head)
        return (logits_deprel, logits_head)

In [87]:
from transformers import AutoConfig

# Config sempre carregado do checkpoint: garante arquitetura correta
MODEL_CONFIG = AutoConfig.from_pretrained(FINETUNED_MODEL_PATH)
TOKENIZER    = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print(f'Modelo        : {MODEL_NAME}  |  Fold: {FOLD}')
print(f'Arquitetura   : {MODEL_CONFIG.model_type} | hidden_size: {MODEL_CONFIG.hidden_size}')
print(f'Checkpoint    : {FINETUNED_MODEL_PATH}')
print(f'Tokenizer     : {TOKENIZER_NAME}')

Modelo        : amadeusai/modernJabuticaBERT-Base-1k  |  Fold: 0
Arquitetura   : modernbert | hidden_size: 768
Checkpoint    : ./best_models_ablacao_linear/amadeusai_modernJabuticaBERT-Base-1k
Tokenizer     : amadeusai/modernJabuticaBERT-Base-1k


In [88]:
# Carregando modelo de ablação (sem UPOS)
model = MultiTaskSentencePredictionEncoderAblacao.from_pretrained(
    FINETUNED_MODEL_PATH,
    config=MODEL_CONFIG,
    num_deprel_labels=len(DEPREL_LABELS),
).to("cuda" if torch.cuda.is_available() else "cpu")
print('Model loaded successfully...')

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 12222.87it/s]

Model loaded successfully...


In [89]:
print(len(DEPREL_LABELS))

44


In [90]:
import pandas as pd
import torch
from tqdm import tqdm

def get_predictions_on_dataframe(sentences, model, tokenizer, device="cpu"):
    predictions_deprel = []
    predictions_head   = []

    probability_deprel = []
    probability_head   = []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        logits_deprel = model_outputs[0]  # cabeça 0: DEPREL
        logits_head   = model_outputs[1]  # cabeça 1: HEAD

        word_ids = inputs.word_ids(batch_index=0)

        sent_preds_deprel, sent_preds_head = [], []
        sent_probs_deprel, sent_probs_head = [], []

        for token_idx in range(len(tokens)):
            subtoken_idxs = [i for i, w_id in enumerate(word_ids) if w_id == token_idx]

            if subtoken_idxs:
                first_sub = subtoken_idxs[0]
                prob_deprel = torch.softmax(logits_deprel[0, first_sub], dim=-1)
                prob_head   = torch.softmax(logits_head[0,   first_sub], dim=-1)

                pred_deprel = torch.argmax(prob_deprel).item()
                pred_head   = torch.argmax(prob_head).item()

                sent_preds_deprel.append(IDX_TO_DEPREL_LABELS[pred_deprel])
                sent_preds_head.append(pred_head)

                sent_probs_deprel.append(prob_deprel[pred_deprel].item())
                sent_probs_head.append(prob_head[pred_head].item())
            else:
                sent_preds_deprel.append(None)
                sent_preds_head.append(None)
                sent_probs_deprel.append(None)
                sent_probs_head.append(None)

        predictions_deprel.append(sent_preds_deprel)
        predictions_head.append(sent_preds_head)
        probability_deprel.append(sent_probs_deprel)
        probability_head.append(sent_probs_head)

    return pd.DataFrame({
        "tokens":             sentences,
        "deprel_predictions": predictions_deprel,
        "deprel_pred_probability": probability_deprel,
        "head_predictions":   predictions_head,
        "head_pred_probability": probability_head,
    })

In [91]:
from datasets import load_from_disk

dataset = load_from_disk('/home/guilhermelima/msc/data_dois/complaints_dataset_obj_outxpos')
test_dataset = dataset['test']
print(f"Test set: {len(test_dataset)} sentenças")

Test set: 1683 sentenças


In [92]:
test_sentences = test_dataset['tokens']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f"Sentenças: {len(test_sentences)}")
print(f"Exemplo tokens : {test_sentences[0]}")
print(f"Exemplo deprel : {test_deprel[0]}")
print(f"Exemplo heads  : {test_head[0]}")

Sentenças: 1683
Exemplo tokens : ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']
Exemplo deprel : ['det', 'nsubj', 'flat:name', 'advmod', 'root', 'det', 'obj', 'punct']
Exemplo heads  : [2, 5, 2, 5, 0, 7, 5, 5]


In [93]:
# test_sentences, test_deprel, test_head já definidos na célula anterior
print(f"Total de sentenças de teste: {len(test_sentences)}")
print(f"Primeira sentença: {test_sentences[0]}")

Total de sentenças de teste: 1683
Primeira sentença: ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [94]:
test_sentences

Column([['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.'], ['A', 'Odebrecht', 'pagou', '300', '%', 'a', 'mais', 'por', 'o', 'direito', 'de', 'explorar', 'o', 'aeroporto', 'de', 'o', 'Galeão', '.'], ['Em', 'o', 'começo', 'de', 'o', 'século', ',', 'a', 'JBS/Friboi', 'chegava', 'a', 'o', 'grupo', 'de', 'as', '400', 'maiores', '.'], ['Os', 'sons', 'indesejáveis', 'emitidos', 'por', 'uma', 'porta', ',', 'por', 'exemplo', ',', 'são', 'eliminados', 'por', 'R$', '150', '.'], ['Que', 'foi', 'herança', 'de', 'o', 'PT', ',', 'que', 'nos', 'deixou', 'esse', 'rombo', ',', 'disse', 'Doria', '.'], ...])

In [95]:
model

MultiTaskSentencePredictionEncoderAblacao(
  (bert): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
  

In [96]:
predict_test_df = get_predictions_on_dataframe(test_sentences, model, TOKENIZER)

100%|██████████| 1683/1683 [01:22<00:00, 20.33it/s]


In [97]:
predict_test_df

,tokens,deprel_predictions,deprel_pred_probability,head_predictions,head_pred_probability
0,"[O, Capitão, América, também, bajulou, o, tuca...","[det, nsubj, flat:name, advmod, root, det, obj...","[0.9999929666519165, 0.9999873638153076, 0.999...","[2, 5, 2, 5, 0, 7, 5, 5]","[0.9999953508377075, 0.9999642372131348, 0.999..."
1,"[A, Odebrecht, pagou, 300, %, a, mais, por, o,...","[det, nsubj, root, nummod, obj, case, advmod, ...","[0.9999982118606567, 0.9999794960021973, 0.999...","[2, 3, 0, 5, 3, 7, 3, 10, 10, 3, 12, 10, 14, 1...","[0.9999855756759644, 0.999990701675415, 0.9999..."
2,"[Em, o, começo, de, o, século, ,, a, JBS/Fribo...","[case, det, obl, case, det, nmod, punct, det, ...","[0.9999932050704956, 0.9999967813491821, 0.999...","[3, 3, 12, 6, 6, 3, 3, 9, 12, 0, 15, 15, 12, 1...","[0.9999964237213135, 0.9999936819076538, 0.999..."
3,"[Os, sons, indesejáveis, emitidos, por, uma, p...","[det, nsubj:pass, amod, acl, case, det, obl:ag...","[0.9999958276748657, 0.999842643737793, 0.9999...","[2, 13, 2, 2, 7, 7, 4, 10, 10, 13, 10, 13, 0, ...","[0.999983549118042, 0.9997989535331726, 0.9999..."
4,"[Que, foi, herança, de, o, PT, ,, que, nos, de...","[nsubj, cop, ccomp, case, det, nmod, punct, ns...","[0.9996716976165771, 0.9999392032623291, 0.992...","[3, 3, 14, 6, 6, 3, 10, 10, 10, 6, 12, 10, 10,...","[0.9997847676277161, 0.9997468590736389, 0.996..."
...,...,...,...,...,...
1678,"[Julia, Louis-Dreyfus, ganhou, por, a, sexta, ...","[nsubj, flat:name, root, case, det, amod, obl,...","[0.9999898672103882, 0.9999886751174927, 0.999...","[3, 1, 0, 7, 7, 7, 3, 7, 10, 3, 13, 13, 3, 16,...","[0.9581841230392456, 0.9990319013595581, 0.999..."
1679,"[Até, a, família, julga, mais, e, apoia, menos...","[advmod, det, nsubj, ccomp:speech, advmod, cc,...","[0.9663147330284119, 0.9999997615814209, 0.999...","[3, 3, 4, 15, 4, 7, 4, 7, 13, 11, 13, 13, 7, 4...","[0.9984927177429199, 0.9999876022338867, 0.999..."
1680,"[Mas, há, episódios, que, indicam, em, Damião,...","[cc, root, obj, nsubj, acl:relcl, case, obl, d...","[0.9999932050704956, 0.9999903440475464, 0.999...","[2, 0, 2, 5, 3, 7, 5, 9, 5, 9, 2]","[0.9998053908348083, 0.9999834299087524, 0.999..."
1681,"["", Mas, Che, permanece, puro, ,, de, certo, m...","[punct, cc, nsubj, ccomp:speech, xcomp, punct,...","[0.9999285936355591, 0.9977892637252808, 0.999...","[4, 4, 4, 13, 4, 9, 9, 9, 4, 9, 4, 13, 0, 13]","[0.9999700784683228, 0.9987314343452454, 0.999..."


In [98]:
for i in range(1):
    print(
        f"Exemplo {i} - Tokens: {len(predict_test_df['tokens'][i])} | "
        f"DEPREL: {len(predict_test_df['deprel_predictions'][i])} | "
        f"HEAD: {len(predict_test_df['head_predictions'][i])}"
    )

Exemplo 0 - Tokens: 8 | DEPREL: 8 | HEAD: 8


In [99]:
print(test_sentences[0])

['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [100]:
def compute_dependency_metrics(test_sentences, test_deprel, test_head, predict_df):
    """
    Métricas para análise de dependências — ablação sem UPOS:
      UAS — Unlabeled Attachment Score: HEAD correto
      LAS — Labeled Attachment Score:   HEAD + DEPREL corretos
    """
    total       = 0
    uas_correct = 0
    las_correct = 0
    skipped     = 0

    for i in range(len(test_sentences)):
        gold_deprel = test_deprel[i]
        gold_head   = test_head[i]

        pred_deprel = predict_df['deprel_predictions'].iloc[i]
        pred_head   = predict_df['head_predictions'].iloc[i]

        for j in range(len(gold_head)):
            if pred_head[j] is None or pred_deprel[j] is None:
                skipped += 1
                continue

            total += 1

            # UAS: HEAD correto
            if pred_head[j] == gold_head[j]:
                uas_correct += 1
                # LAS: HEAD correto E DEPREL correto
                if pred_deprel[j] == gold_deprel[j]:
                    las_correct += 1

    return {
        'uas':          uas_correct / total if total > 0 else 0,
        'las':          las_correct / total if total > 0 else 0,
        'total_tokens': total,
        'skipped':      skipped,
    }

In [101]:
metrics = compute_dependency_metrics(
    test_sentences, test_deprel, test_head, predict_test_df
)

print(f"UAS           : {metrics['uas']:.4f}")
print(f"LAS           : {metrics['las']:.4f}")
print(f"Total tokens  : {metrics['total_tokens']}")
print(f"Ignorados     : {metrics['skipped']}")

UAS           : 0.8700
LAS           : 0.8529
Total tokens  : 33580
Ignorados     : 0


In [102]:
#text = [["O", "gato", "preto", "dorme", "no", "sofá", "."]]
#text = [["A", "menina", "brinca", "no", "parque", "."]]
#text = [["Se", "chover", ",", "o", "jogo", "será", "cancelado", "."]]
text = [['Mas', 'por', 'não', 'existir', 'um', 'marco', 'legal', 'há', 'uma',
         'insegurança', 'por', 'parte', 'dos', 'investidores', '"', ',', 'destacou', '.']]

In [103]:
retorno = get_predictions_on_dataframe(text, model, TOKENIZER)

100%|██████████| 1/1 [00:00<00:00, 20.03it/s]


In [104]:
retorno

,tokens,deprel_predictions,deprel_pred_probability,head_predictions,head_pred_probability
0,"[Mas, por, não, existir, um, marco, legal, há,...","[cc, mark, advmod, advcl, det, nsubj, amod, cc...","[0.9999620914459229, 0.9994383454322815, 0.999...","[8, 4, 4, 8, 6, 4, 6, 17, 10, 8, 12, 8, 14, 12...","[0.9998437166213989, 0.9999176263809204, 0.999..."


In [105]:
print('Gold deprel sentença 2:', test_deprel[2])
print('Pred deprel sentença 2:', predict_test_df['deprel_predictions'].iloc[2])

Gold deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']
Pred deprel sentença 2: ['case', 'det', 'obl', 'case', 'det', 'nmod', 'punct', 'det', 'nsubj', 'root', 'case', 'det', 'obl', 'case', 'det', 'nmod', 'amod', 'punct']


In [106]:
out_csv = f'./predict_test_ablacao_linear_{MODEL_NAME.replace("/", "_")}_fold{FOLD}.csv'
predict_test_df.to_csv(out_csv, index=False)
print(f"Salvo em: {out_csv}")

Salvo em: ./predict_test_ablacao_linear_amadeusai_modernJabuticaBERT-Base-1k_fold0.csv
